In [ ]:
!pip install biopython -q

import random
import json
from datetime import datetime
from Bio import SeqIO
from Bio.Seq import Seq
from Bio.SeqRecord import SeqRecord

print("Step 1: Generating Realistic DNA Data ")

#  CONFIGURATION
random.seed(42)
NUM_SAMPLES       = 2000
SEQ_LENGTH        = 150
FASTA_FILENAME    = "alzheimers_simulated.fasta"
METADATA_FILENAME = "dataset_metadata.json"

# AD markers
AD_MARKERS = {
    "APOE4_rs429358": {"seq": "TCCG", "significance": "Pathogenic"},
    "APOE4_rs7412":   {"seq": "CTCA", "significance": "Pathogenic"},
}

#  REFERENCE GENOME
random.seed(0)
REFERENCE_GENOME = "".join(random.choices(['A','C','G','T'], k=SEQ_LENGTH))
random.seed(42)

#  ALIGNMENT SIMULATION
def simulate_alignment(sequence_str, patient_id):
    """
    Compares the generated sequence with the reference genome
    to extract variants dynamically.
    """
    variants = []
    pathogenic_count = 0

    # Simple simulation to detect differences
    for i in range(SEQ_LENGTH):
        if sequence_str[i] != REFERENCE_GENOME[i]:
            variants.append({"position": i, "alt": sequence_str[i]})

    # Search for Alzheimer's markers anywhere in the sequence
    for marker_name, marker_info in AD_MARKERS.items():
        if marker_info["seq"] in sequence_str:
            pathogenic_count += 1

    return {
        "total_variants": len(variants),
        "pathogenic_count": pathogenic_count
    }

# DATA GENERATION
records = []
patient_registry = []

print("Generating sequences with random mutation placements...")

for i in range(NUM_SAMPLES):
    is_ad = i % 2 == 0
    status = "AD" if is_ad else "CTL"

    # Start from the reference genome
    seq_list = list(REFERENCE_GENOME)

    # 1. Add natural noise (Benign Variants) for all samples
    for _ in range(random.randint(2, 8)):
        rand_pos = random.randint(0, SEQ_LENGTH - 1)
        seq_list[rand_pos] = random.choice(['A', 'C', 'G', 'T'])

    # 2. Inject Alzheimer's mutations (at random positions) for AD patients only
    if is_ad:
        for marker_name, marker_info in AD_MARKERS.items():
            if random.random() < 0.85: # 85% probability of having the marker
                marker_seq = list(marker_info["seq"])
                # Choose a random starting position for the mutation
                insert_pos = random.randint(0, SEQ_LENGTH - len(marker_seq))
                seq_list[insert_pos:insert_pos + len(marker_seq)] = marker_seq

    seq_str = "".join(seq_list)
    patient_id = f"Patient_{i:04d}"

    # Evaluate mutations
    alignment = simulate_alignment(seq_str, patient_id)

    # Log GDPR metadata (Crucial for the unlearning phase later)
    patient_registry.append({
        "patient_id": patient_id,
        "label": 1 if is_ad else 0,
        "pathogenic_variants": alignment["pathogenic_count"],
        "contribution_date": datetime.now().isoformat()
    })

    # Create FASTA record
    record = SeqRecord(
        Seq(seq_str),
        id=patient_id,
        description=f"Status:{status}|Variants:{alignment['total_variants']}"
    )
    records.append(record)

# Save the FASTA file
SeqIO.write(records, FASTA_FILENAME, "fasta")

# Save the Metadata JSON
metadata = {
    "dataset_info": {"num_samples": NUM_SAMPLES, "seq_length": SEQ_LENGTH},
    "gdpr_compliance": {
        "purpose": "Alzheimer's disease genetic risk assessment & SISA Unlearning",
        "patient_registry": patient_registry
    }
}

with open(METADATA_FILENAME, "w") as f:
    json.dump(metadata, f, indent=2)

print(f" Total Patients  : {NUM_SAMPLES}")
print(f" GDPR Audit      : {len(patient_registry)} patients registered")


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 58.4 MB/s eta 0:00:00
Step 1: Generating Realistic DNA Data 
Generating sequences with random mutation placements...
 Total Patients  : 2000
 GDPR Audit      : 2000 patients registered


In [ ]:
# : K-mer Extraction & Vectorization (Text to Numerical)
import numpy as np
import pickle
from Bio import SeqIO
from sklearn.feature_extraction.text import CountVectorizer, TfidfTransformer

print("Step 2: K-mer Extraction (Text to Vectors using TF-IDF)")

#  CONFIGURATION
FASTA_FILENAME = "alzheimers_simulated.fasta"
KMER_SIZE      = 4
SEQ_LENGTH     = 150

# VALIDATION
assert KMER_SIZE <= SEQ_LENGTH, (
    f"Error: k-mer size ({KMER_SIZE}) cannot exceed sequence length ({SEQ_LENGTH})"
)

print(f"\nConfiguration:")
print(f" - FASTA File            : {FASTA_FILENAME}")
print(f" - K-mer Size            : {KMER_SIZE}")
print(f" - K-mers per sequence   : {SEQ_LENGTH - KMER_SIZE + 1}")

#  K-MER EXTRACTION
def get_kmers(sequence, k=4):
    """
    Slices a DNA sequence into overlapping k-mers.
    Example: "ATCG" with k=2 -> ["AT", "TC", "CG"]
    """
    return [sequence[i:i+k] for i in range(len(sequence) - k + 1)]

#  PARSE FASTA
parsed_sequences = []
y_labels         = []
patient_ids      = []

print("\nReading FASTA file and extracting k-mers...")

for record in SeqIO.parse(FASTA_FILENAME, "fasta"):
    patient_id = record.id
    status = "AD" if "Status:AD" in record.description else "CTL"
    label = 1 if status == "AD" else 0

    patient_ids.append(patient_id)
    y_labels.append(label)

    sequence_str = str(record.seq).upper()
    kmers_list   = get_kmers(sequence_str, k=KMER_SIZE)
    # Join k-mers with space so CountVectorizer can process them like words
    parsed_sequences.append(" ".join(kmers_list))

print(f" - Total sequences parsed : {len(parsed_sequences)}")
print(f" - AD samples             : {sum(y_labels)}")
print(f" - CTL samples            : {len(y_labels) - sum(y_labels)}")

#  VECTORIZATION WITH TF-IDF
print("\nConverting k-mer sentences to numerical vectors (TF-IDF)...")

# Step A: Count frequencies (min_df=2 removes extremely rare noise)
count_vectorizer = CountVectorizer(
    analyzer='word',
    token_pattern=r'\S+',  # Treat each k-mer as one token
    min_df=2
)
X_counts = count_vectorizer.fit_transform(parsed_sequences)

# Step B: Apply TF-IDF normalization
# Rare k-mers (pathogenic markers) get higher weight
tfidf_transformer = TfidfTransformer()
X_features = tfidf_transformer.fit_transform(X_counts).toarray()

y_target      = np.array(y_labels)
feature_names = count_vectorizer.get_feature_names_out()
vocab_size    = len(feature_names)

print(f" - Vocabulary size        : {vocab_size} unique {KMER_SIZE}-mers")
print(f" - Feature matrix shape   : {X_features.shape}")

#  SAVE PROCESSED DATA (Crucial for SISA)
print("\nSaving processed data")

# Save matrices
np.save("X_kmers.npy",     X_features)
np.save("y_labels.npy",    y_target)
np.save("patient_ids.npy", np.array(patient_ids)) # Key for GDPR deletion

# Save models
with open("kmer_vectorizer.pkl", "wb") as f:
    pickle.dump(count_vectorizer, f)

with open("tfidf_transformer.pkl", "wb") as f:
    pickle.dump(tfidf_transformer, f)

print(" SUCCESS - K-mer Extraction & TF-IDF Complete")

Step 2: K-mer Extraction (Text to Vectors using TF-IDF)

Configuration:
 - FASTA File            : alzheimers_simulated.fasta
 - K-mer Size            : 4
 - K-mers per sequence   : 147

Reading FASTA file and extracting k-mers...
 - Total sequences parsed : 2000
 - AD samples             : 1000
 - CTL samples            : 1000

Converting k-mer sentences to numerical vectors (TF-IDF)...
 - Vocabulary size        : 256 unique 4-mers
 - Feature matrix shape   : (2000, 256)

Saving processed data
 SUCCESS - K-mer Extraction & TF-IDF Complete


In [ ]:
# = Bioinformatics Validation (Sequence Alignment)
import warnings
from Bio import BiopythonDeprecationWarning

warnings.simplefilter('ignore', BiopythonDeprecationWarning)

from Bio import pairwise2
from Bio.pairwise2 import format_alignment

print("= Validating Mutations via Sequence Alignment")


seq_ctl = "ATGCGTACGTAGCTAGCTAGCTAGCTAGCTAGCTAGCTAGCTAGCTAGCT"
seq_ad  = "ATGCGTACGTAGCTAGCTAGCTAGCGAGCTAGCTAGCTAGCTAGCTAGCA"

print("Aligning AD Patient Sample vs Control Person Sample...\n")


alignments = pairwise2.align.globalxx(seq_ctl, seq_ad)

for alignment in alignments[:1]:
    print(format_alignment(*alignment))

= Validating Mutations via Sequence Alignment
Aligning AD Patient Sample vs Control Person Sample...

ATGCGTACGTAGCTAGCTAGCTAGCT-AGCTAGCTAGCTAGCTAGCTAGCT-
|||||||||||||||||||||||||  |||||||||||||||||||||||  
ATGCGTACGTAGCTAGCTAGCTAGC-GAGCTAGCTAGCTAGCTAGCTAGC-A
  Score=48



In [ ]:
# Building and Training the CNN Model
import os
import time
import numpy as np
import tensorflow as tf
from tensorflow.keras import layers, models, metrics, Input
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import classification_report, roc_auc_score, roc_curve
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

print("Step 3: Building and Training the Baseline CNN Model ")

# CONFIGURATION & SEED
tf.random.set_seed(42)
np.random.seed(42)

#  LOAD DATA
X = np.load("X_kmers.npy")
y = np.load("y_labels.npy")

print(f" - Feature matrix : {X.shape}")
print(f" - AD={sum(y)}, CTL={len(y)-sum(y)}")

# Reshape for Conv1D
X_cnn = X.reshape((X.shape[0], X.shape[1], 1))

#  TRAIN/TEST SPLIT
X_train, X_test, y_train, y_test = train_test_split(
    X_cnn, y, test_size=0.2, random_state=42, stratify=y
)
print(f" - Train: {X_train.shape[0]} | Test: {X_test.shape[0]}")

class_weights = compute_class_weight('balanced', classes=np.unique(y_train), y=y_train)
class_weight_dict = {i: class_weights[i] for i in range(len(class_weights))}

#  BUILD CNN ARCHITECTURE
def build_cnn_model(input_shape):
    inputs = Input(shape=input_shape)

    x = layers.Conv1D(32, 3, activation='relu', padding='same')(inputs)
    x = layers.BatchNormalization()(x)
    x = layers.MaxPooling1D(2)(x)
    x = layers.Dropout(0.2)(x)

    x = layers.Conv1D(64, 3, activation='relu', padding='same')(x)
    x = layers.BatchNormalization()(x)
    x = layers.MaxPooling1D(2)(x)
    x = layers.Dropout(0.2)(x)

    x = layers.Conv1D(128, 3, activation='relu', padding='same')(x)
    x = layers.BatchNormalization()(x)
    x = layers.GlobalMaxPooling1D()(x)

    x = layers.Dense(128, activation='relu')(x)
    x = layers.Dropout(0.4)(x)
    x = layers.Dense(64, activation='relu')(x)
    x = layers.Dropout(0.3)(x)
    outputs = layers.Dense(1, activation='sigmoid')(x)

    return models.Model(inputs, outputs)

model = build_cnn_model((X_cnn.shape[1], 1))

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss='binary_crossentropy',
    metrics=['accuracy', metrics.AUC(name='auc')]
)

# CALLBACKS
early_stop = tf.keras.callbacks.EarlyStopping(
    monitor='val_auc',
    patience=15,
    mode='max',
    restore_best_weights=True,
    verbose=1
)

lr_scheduler = tf.keras.callbacks.ReduceLROnPlateau(
    monitor='val_auc',
    factor=0.5,
    patience=5,
    mode='max',
    min_lr=1e-6,
    verbose=1
)

# TRAIN THE MODEL
print("\nTraining Started")
start_time = time.time()

history = model.fit(
    X_train, y_train,
    epochs=50,
    batch_size=32,
    validation_data=(X_test, y_test),
    class_weight=class_weight_dict,
    callbacks=[early_stop, lr_scheduler],
    verbose=1
)

train_time = time.time() - start_time
epochs_trained = len(history.history['loss'])
print(f"\n Epochs: {epochs_trained} | Time: {train_time:.2f}s")

#  SMART THRESHOLD TUNING
print("\nCalculating Optimal Threshold")
y_pred_proba = model.predict(X_test).flatten()

fpr, tpr, thresholds = roc_curve(y_test, y_pred_proba)
optimal_idx = np.argmax(tpr - fpr)
best_threshold = thresholds[optimal_idx]

y_pred_final = (y_pred_proba >= best_threshold).astype(int)
best_acc = (y_pred_final == y_test).mean()
final_auc = roc_auc_score(y_test, y_pred_proba)

print(f"  Best Auto-Threshold : {best_threshold:.4f}")
print(f"  Best Accuracy       : {best_acc*100:.1f}%")

#  EVALUATION
print("\nFinal Results:")
print(f"  AUC      : {final_auc:.4f}")
print(f"  Accuracy : {best_acc*100:.2f}%")
print(classification_report(y_test, y_pred_final, target_names=['CTL','AD']))

#  PLOTS
plt.figure(figsize=(15, 5))

plt.subplot(1, 2, 1)
plt.plot(history.history['accuracy'],     label='Train', linewidth=2)
plt.plot(history.history['val_accuracy'], label='Val',   linewidth=2)
plt.title('Accuracy')
plt.legend()
plt.grid(True, alpha=0.3)

plt.subplot(1, 2, 2)
plt.plot(history.history['auc'],     label='Train AUC', linewidth=2)
plt.plot(history.history['val_auc'], label='Val AUC',   linewidth=2)
plt.title('AUC')
plt.legend()
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('learning_curves.png', dpi=150, bbox_inches='tight')

#  SAVE METRICS & MODEL
model.save("baseline_cnn_model.keras")
model_size_mb = os.path.getsize("baseline_cnn_model.keras") / (1024 * 1024)

baseline_metrics = {
    "model_type":        "CNN",
    "epochs_trained":    int(epochs_trained),
    "training_time_sec": float(train_time),
    "best_threshold":    float(best_threshold),
    "best_accuracy":     float(best_acc),
    "test_auc":          float(final_auc),
    "model_size_mb":     float(model_size_mb)
}

with open("baseline_metrics.json", "w") as f:
    json.dump(baseline_metrics, f, indent=2)

print(f"\n Model saved ({model_size_mb:.2f} MB)")

Step 3: Building and Training the Baseline CNN Model 
 - Feature matrix : (2000, 256)
 - AD=1000, CTL=1000
 - Train: 1600 | Test: 400

Training Started
Epoch 1/50
50/50 ━━━━━━━━━━━━━━━━━━━━ 17s 96ms/step - accuracy: 0.5206 - auc: 0.5120 - loss: 1.0952 - val_accuracy: 0.5000 - val_auc: 0.5275 - val_loss: 0.6936 - learning_rate: 0.0010
Epoch 2/50
50/50 ━━━━━━━━━━━━━━━━━━━━ 3s 68ms/step - accuracy: 0.5100 - auc: 0.5244 - loss: 0.7290 - val_accuracy: 0.5000 - val_auc: 0.4927 - val_loss: 0.6963 - learning_rate: 0.0010
Epoch 3/50
50/50 ━━━━━━━━━━━━━━━━━━━━ 5s 90ms/step - accuracy: 0.5281 - auc: 0.5396 - loss: 0.7072 - val_accuracy: 0.5000 - val_auc: 0.5930 - val_loss: 0.7003 - learning_rate: 0.0010
Epoch 4/50
50/50 ━━━━━━━━━━━━━━━━━━━━ 5s 94ms/step - accuracy: 0.5412 - auc: 0.5682 - loss: 0.6941 - val_accuracy: 0.5000 - val_auc: 0.6391 - val_loss: 0.6995 - learning_rate: 0.0010
Epoch 5/50
50/50 ━━━━━━━━━━━━━━━━━━━━ 4s 84ms/step - accuracy: 0.5881 - auc: 0.6138 - loss: 0.6760 - val_accuracy: 

In [ ]:
#  SISA Implementation (Sharding & Isolated Training)

print("Step 4: Implementing SISA (Sharded & Isolated Training)")

#  CONFIGURATION
NUM_SHARDS = 5
EPOCHS_PER_SHARD = 40

# LOAD DATA
# Loading the data saved from Cell 2
X_sisa = np.load("X_kmers.npy")
y_sisa = np.load("y_labels.npy")
patient_ids_sisa = np.load("patient_ids.npy")

# Reshape for Conv1D
X_cnn_sisa = X_sisa.reshape((X_sisa.shape[0], X_sisa.shape[1], 1))
total_samples = len(y_sisa)

print(f"Total Samples: {total_samples} | Splitting into {NUM_SHARDS} Shards...")

# SHARDING LOGIC
# Shuffle the data first to ensure random distribution across shards
indices = np.arange(total_samples)
np.random.shuffle(indices)

X_shuffled = X_cnn_sisa[indices]
y_shuffled = y_sisa[indices]
patients_shuffled = patient_ids_sisa[indices]

# Split into equal shards
X_shards = np.array_split(X_shuffled, NUM_SHARDS)
y_shards = np.array_split(y_shuffled, NUM_SHARDS)
patient_shards = np.array_split(patients_shuffled, NUM_SHARDS)

# Create a mapping registry: Which patient is in which shard?
# This is CRITICAL for the "Unlearning" phase later
patient_to_shard_map = {}
for shard_id, p_list in enumerate(patient_shards):
    for pid in p_list:
        patient_to_shard_map[pid] = int(shard_id)

# Save the registry so we can look up patients when they request deletion
with open("sisa_patient_registry.json", "w") as f:
    json.dump(patient_to_shard_map, f, indent=2)

print(" Data successfully sharded and Patient Registry saved.")

#  BUILD SHARD MODEL
# Same architecture as baseline, redefined here for clarity and isolation
def build_shard_model(input_shape):
    inputs = Input(shape=input_shape)

    x = layers.Conv1D(32, 3, activation='relu', padding='same')(inputs)
    x = layers.BatchNormalization()(x)
    x = layers.MaxPooling1D(2)(x)
    x = layers.Dropout(0.2)(x)

    x = layers.Conv1D(64, 3, activation='relu', padding='same')(x)
    x = layers.BatchNormalization()(x)
    x = layers.MaxPooling1D(2)(x)
    x = layers.Dropout(0.2)(x)

    x = layers.Conv1D(128, 3, activation='relu', padding='same')(x)
    x = layers.BatchNormalization()(x)
    x = layers.GlobalMaxPooling1D()(x) # Catching mutations

    x = layers.Dense(128, activation='relu')(x)
    x = layers.Dropout(0.4)(x)
    x = layers.Dense(64, activation='relu')(x)
    x = layers.Dropout(0.3)(x)
    outputs = layers.Dense(1, activation='sigmoid')(x)

    model = models.Model(inputs, outputs)
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
        loss='binary_crossentropy',
        metrics=['accuracy', metrics.AUC(name='auc')]
    )
    return model

#  TRAIN EACH SHARD ISOLATED
# Ensure directory exists for saving shard models
os.makedirs("sisa_models", exist_ok=True)
shard_metrics = {}

print("\nStarting Isolated Training for each Shard")
total_sisa_train_time = 0

for shard_id in range(NUM_SHARDS):
    print(f"\n--- Training Shard {shard_id + 1}/{NUM_SHARDS} ---")
    X_s = X_shards[shard_id]
    y_s = y_shards[shard_id]

    # Train/Test split WITHIN the specific shard
    X_train_s, X_test_s, y_train_s, y_test_s = train_test_split(
        X_s, y_s, test_size=0.2, random_state=42, stratify=y_s
    )

    # Balance classes for this specific shard
    class_weights = compute_class_weight('balanced', classes=np.unique(y_train_s), y=y_train_s)
    cw_dict = {i: class_weights[i] for i in range(len(class_weights))}

    early_stop = tf.keras.callbacks.EarlyStopping(
        monitor='val_auc', patience=10, mode='max', restore_best_weights=True, verbose=0
    )

    model_s = build_shard_model((X_s.shape[1], 1))

    start_time_s = time.time()

    # verbose=0 to keep logs clean, we only print final shard times
    history_s = model_s.fit(
        X_train_s, y_train_s,
        epochs=EPOCHS_PER_SHARD,
        batch_size=16, # Smaller batch size because we have less data per shard
        validation_data=(X_test_s, y_test_s),
        class_weight=cw_dict,
        callbacks=[early_stop],
        verbose=0
    )

    shard_time = time.time() - start_time_s
    total_sisa_train_time += shard_time

    # Save the specific shard model
    model_path = f"sisa_models/shard_{shard_id}.keras"
    model_s.save(model_path)

    # Extract the best AUC reached
    val_auc = max(history_s.history['val_auc'])
    print(f" Shard {shard_id} completed in {shard_time:.2f}s | Best Val AUC: {val_auc:.4f}")

    # Log metrics
    shard_metrics[f"shard_{shard_id}"] = {
        "train_time": float(shard_time),
        "val_auc": float(val_auc),
        "samples": len(y_s)
    }

#  SAVE SISA METRICS
sisa_summary = {
    "num_shards": NUM_SHARDS,
    "total_sisa_train_time": float(total_sisa_train_time),
    "average_time_per_shard": float(total_sisa_train_time / NUM_SHARDS),
    "shard_details": shard_metrics
}

with open("sisa_metrics.json", "w") as f:
    json.dump(sisa_summary, f, indent=2)

print("\n SISA Initial Training Complete!")
print(f"Total time to train all {NUM_SHARDS} shards: {total_sisa_train_time:.2f}s")
print(f"Average time per shard (Cost of unlearning later): {sisa_summary['average_time_per_shard']:.2f}s")

Step 4: Implementing SISA (Sharded & Isolated Training)
Total Samples: 2000 | Splitting into 5 Shards...
 Data successfully sharded and Patient Registry saved.

Starting Isolated Training for each Shard

--- Training Shard 1/5 ---
 Shard 0 completed in 11.72s | Best Val AUC: 0.5122

--- Training Shard 2/5 ---
 Shard 1 completed in 34.99s | Best Val AUC: 0.9294

--- Training Shard 3/5 ---
 Shard 2 completed in 27.61s | Best Val AUC: 0.9624

--- Training Shard 4/5 ---
 Shard 3 completed in 27.77s | Best Val AUC: 0.9453

--- Training Shard 5/5 ---
 Shard 4 completed in 26.73s | Best Val AUC: 0.9475

 SISA Initial Training Complete!
Total time to train all 5 shards: 128.82s
Average time per shard (Cost of unlearning later): 25.76s


In [ ]:
#  Machine Unlearning Simulation (Random GDPR Deletion Request)
import random

print("Step 5: Simulating SISA Unlearning (GDPR Deletion Request)")

# 1. Load the SISA Registry and Baseline Metadata
with open("sisa_patient_registry.json", "r") as f:
    registry = json.load(f)

with open("baseline_metrics.json", "r") as f:
    baseline_metrics = json.load(f)

# 2. Simulate a Deletion Request (Pick a TRULY random patient every time)
target_patient = random.choice(list(registry.keys()))
target_shard = registry[target_patient]

print(f"\n UNLEARNING REQUEST RECEIVED:")
print(f" - Patient ID : {target_patient}")
print(f" - Located in : Shard {target_shard}")

# 3. Isolate the target Shard's Data and REMOVE the patient
X_full = np.load("X_kmers.npy")
y_full = np.load("y_labels.npy")
pids_full = np.load("patient_ids.npy")

X_cnn_full = X_full.reshape((X_full.shape[0], X_full.shape[1], 1))

# Find indices of all patients in this shard EXCEPT the target patient
shard_indices = []
for idx, pid in enumerate(pids_full):
    if registry.get(pid) == target_shard and pid != target_patient:
        shard_indices.append(idx)

X_unlearn = X_cnn_full[shard_indices]
y_unlearn = y_full[shard_indices]

print(f"\nData for Shard {target_shard} prepared.")
print(f" - Original Shard Size : {len(shard_indices) + 1}")
print(f" - New Shard Size      : {len(shard_indices)} (Patient eradicated!)")

# 4. Retrain ONLY the affected Shard
def build_unlearn_model(input_shape):
    inputs = Input(shape=input_shape)
    x = layers.Conv1D(32, 3, activation='relu', padding='same')(inputs)
    x = layers.BatchNormalization()(x)
    x = layers.MaxPooling1D(2)(x)
    x = layers.Dropout(0.2)(x)

    x = layers.Conv1D(64, 3, activation='relu', padding='same')(x)
    x = layers.BatchNormalization()(x)
    x = layers.MaxPooling1D(2)(x)
    x = layers.Dropout(0.2)(x)

    x = layers.Conv1D(128, 3, activation='relu', padding='same')(x)
    x = layers.BatchNormalization()(x)
    x = layers.GlobalMaxPooling1D()(x)

    x = layers.Dense(128, activation='relu')(x)
    x = layers.Dropout(0.4)(x)
    x = layers.Dense(64, activation='relu')(x)
    x = layers.Dropout(0.3)(x)
    outputs = layers.Dense(1, activation='sigmoid')(x)

    model = models.Model(inputs, outputs)
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
        loss='binary_crossentropy',
        metrics=['accuracy', metrics.AUC(name='auc')]
    )
    return model

print(f"\nRetraining Shard {target_shard} from scratch...")

X_train_u, X_test_u, y_train_u, y_test_u = train_test_split(
    X_unlearn, y_unlearn, test_size=0.2, random_state=42, stratify=y_unlearn
)

class_weights = compute_class_weight('balanced', classes=np.unique(y_train_u), y=y_train_u)
cw_dict = {i: class_weights[i] for i in range(len(class_weights))}

early_stop = tf.keras.callbacks.EarlyStopping(
    monitor='val_auc', patience=10, mode='max', restore_best_weights=True, verbose=0
)

unlearn_model = build_unlearn_model((X_unlearn.shape[1], 1))

start_time = time.time()

# Retraining (verbose=0 to keep it clean)
history = unlearn_model.fit(
    X_train_u, y_train_u,
    epochs=40,
    batch_size=16,
    validation_data=(X_test_u, y_test_u),
    class_weight=cw_dict,
    callbacks=[early_stop],
    verbose=0
)

unlearning_time = time.time() - start_time
new_val_auc = max(history.history['val_auc'])

# Save the newly retrained shard (replacing the old one)
unlearn_model.save(f"sisa_models/shard_{target_shard}_unlearned.keras")

print(f" Retraining complete in {unlearning_time:.2f} seconds!")
print(f" - New Validation AUC: {new_val_auc:.4f}")

# 5. GENERATE FINAL COMPARISON REPORT
baseline_time = baseline_metrics["training_time_sec"]
time_saved = baseline_time - unlearning_time
speedup_factor = baseline_time / unlearning_time

print(" FINAL MACHINE UNLEARNING PERFORMANCE REPORT ")
print("="*55)
print(f"1. Traditional Method (Full Retrain) : {baseline_time:.2f} seconds")
print(f"2. SISA Method (Targeted Retrain)    : {unlearning_time:.2f} seconds")
print("-" * 55)
print(f" Time Saved                        : {time_saved:.2f} seconds")
print(f" Speedup Factor                    : {speedup_factor:.1f}x faster!")
print(f" GDPR Compliance                   : 100% Confirmed")


Step 5: Simulating SISA Unlearning (GDPR Deletion Request)

 UNLEARNING REQUEST RECEIVED:
 - Patient ID : Patient_0548
 - Located in : Shard 1

Data for Shard 1 prepared.
 - Original Shard Size : 400
 - New Shard Size      : 399 (Patient eradicated!)

Retraining Shard 1 from scratch...
 Retraining complete in 37.25 seconds!
 - New Validation AUC: 0.9394
 FINAL MACHINE UNLEARNING PERFORMANCE REPORT 
1. Traditional Method (Full Retrain) : 135.52 seconds
2. SISA Method (Targeted Retrain)    : 37.25 seconds
-------------------------------------------------------
 Time Saved                        : 98.27 seconds
 Speedup Factor                    : 3.6x faster!
 GDPR Compliance                   : 100% Confirmed


In [ ]:
# Membership Inference Attack (MIA) & Confidence Analysis
import numpy as np

print("Step 6: Executing Membership Inference Attack (MIA)")

def membership_inference_attack(model, X_train, X_test, test_patient_id):
    train_preds = model.predict(X_train, verbose=0)
    test_preds = model.predict(X_test, verbose=0)


    train_conf = np.array([max(p[0], 1 - p[0]) for p in train_preds])
    test_conf = np.array([max(p[0], 1 - p[0]) for p in test_preds])

    threshold = np.mean(train_conf)

    train_infer = train_conf > threshold
    test_infer = test_conf > threshold

    print(f"Calculated Confidence Threshold: {threshold:.4f}")
    print("-" * 40)
    print(f"Train samples correctly inferred as members: {np.sum(train_infer)} / {len(train_infer)}")

    is_member = test_infer[0]
    print(f"Target Patient ({test_patient_id}) inferred as member: {is_member}")

    if not is_member:
        print(" ATTACK FAILED: The model has successfully FORGOTTEN the patient.")
        print("The deleted patient's confidence is now below the membership threshold.")
        print("PRIVACY STATUS: COMPLIANT (GDPR Safe)")
    else:
        print(" ATTACK SUCCESSFUL: The model still shows signs of remembering this patient.")
        print("PRIVACY STATUS: VULNERABLE")

target_idx = np.where(pids_full == target_patient)[0][0]
X_deleted_sample = X_cnn_full[target_idx:target_idx+1]

membership_inference_attack(unlearn_model, X_train_u, X_deleted_sample, target_patient)



Step 6: Executing Membership Inference Attack (MIA)
Calculated Confidence Threshold: 0.9386
----------------------------------------
Train samples correctly inferred as members: 221 / 319
Target Patient (Patient_0548) inferred as member: False
 ATTACK FAILED: The model has successfully FORGOTTEN the patient.
The deleted patient's confidence is now below the membership threshold.
PRIVACY STATUS: COMPLIANT (GDPR Safe)
